# EKATENA – Extracción de coordenadas de un texto

**Cómo usarlo**
0. La primera vez, instala la librería para leer PDF ejecutando en la terminal: `pip install pypdf`
1. En la celda **1. Datos de entrada**, escribe la ruta de tu PDF en `RUTA_PDF` (o deja `RUTA_PDF = ""` y pega el texto en `TEXTO`).
2. Si quieres buscar coordenadas solo después de una frase concreta (por ejemplo `"Coordenadas del punto:"`), escríbela en `BUSCAR`. Déjalo vacío (`""`) para buscar en todo el texto.
3. Ejecuta todas las celdas (*Run All*).

**Formatos que reconoce**
- Decimal: `19.4326, -99.1332`
- Grados, minutos y segundos (GMS): `19°25'57.3" N 99°07'59.5" O` (N/S/E/O/W o Norte/Sur/Este/Oeste)
- UTM: `14Q 483000 2147000` (se extrae; no se convierte a lat/lon)

In [ ]:
# === 1. Datos de entrada ===

# Ruta del PDF. Usa r"..." para que funcionen las barras invertidas de Windows.
# Si lo dejas vacío (""), se usa el TEXTO de abajo.
RUTA_PDF = r""
# Ejemplo: RUTA_PDF = r"C:\Users\tu_usuario\Desktop\documento.pdf"
# Si el PDF está en la misma carpeta que este notebook, basta con el nombre: r"documento.pdf"

TEXTO = """
Pega aquí tu texto. Ejemplo:
Coordenadas del punto: 19.4326, -99.1332
Otro punto en 20°40'36.5" N 103°20'51.2" O.
Coordenadas del punto: 25.6866, -100.3161
Ubicación UTM: 14Q 483000 2147000
"""

# Texto exacto a buscar. Si lo dejas vacío (""), se busca en todo el texto.
BUSCAR = "Coordenadas del punto:"

# Cuántos caracteres después de BUSCAR se revisan para encontrar la coordenada
VENTANA = 200

In [ ]:
# === 2. Funciones de extracción ===
import re

NUM_DEC = r"[-+]?\d{1,3}\.\d+"
RE_DECIMAL = re.compile(
    rf"(?<![\d.])({NUM_DEC})\s*[,;]?\s+({NUM_DEC})(?![\d.])"
    rf"|(?<![\d.])({NUM_DEC})\s*[,;]\s*({NUM_DEC})(?![\d.])"
)

HEMI = r"(Norte|Sur|Este|Oeste|[NSEOW])\b"
RE_GMS = re.compile(
    r"(\d{1,3})\s*[°º]\s*"                     # grados
    r"(?:(\d{1,2}(?:[.,]\d+)?)\s*['′’]\s*)?"   # minutos (opcional)
    r"(?:(\d{1,2}(?:[.,]\d+)?)\s*(?:\"|″|”|''|’’)\s*)?"  # segundos (opcional)
    rf"(?:{HEMI})?",
    re.IGNORECASE,
)

RE_UTM = re.compile(
    r"\b(\d{1,2})\s*([C-HJ-NP-X])\s+(\d{6}(?:\.\d+)?)\s*m?\s*E?\s*[,;]?\s*(\d{7}(?:\.\d+)?)\s*m?\s*N?\b"
)


def _num(s):
    return float(s.replace(",", ".")) if s else 0.0


def gms_a_decimal(g, m, s, hemisferio):
    valor = _num(g) + _num(m) / 60 + _num(s) / 3600
    if hemisferio and hemisferio[0].upper() in ("S", "O", "W"):
        valor = -valor
    return round(valor, 6)


def _es_lat(h):
    return h and h[0].upper() in ("N", "S")


def extraer_coordenadas(texto):
    """Devuelve una lista de diccionarios con las coordenadas encontradas en `texto`."""
    resultados = []

    # Decimal
    for m in RE_DECIMAL.finditer(texto):
        a, b = (m.group(1), m.group(2)) if m.group(1) else (m.group(3), m.group(4))
        lat, lon = float(a), float(b)
        if -90 <= lat <= 90 and -180 <= lon <= 180:
            resultados.append({"formato": "decimal", "texto": m.group(0).strip(),
                               "lat": lat, "lon": lon, "pos": m.start()})

    # GMS: se emparejan de dos en dos (latitud, longitud)
    gms = [m for m in RE_GMS.finditer(texto)]
    i = 0
    while i + 1 < len(gms):
        m1, m2 = gms[i], gms[i + 1]
        v1 = gms_a_decimal(*m1.groups())
        v2 = gms_a_decimal(*m2.groups())
        h1, h2 = m1.group(4), m2.group(4)
        if _es_lat(h2) and not _es_lat(h1):      # vienen como lon, lat
            v1, v2 = v2, v1
        resultados.append({"formato": "GMS", "texto": texto[m1.start():m2.end()].strip(),
                           "lat": v1, "lon": v2, "pos": m1.start()})
        i += 2

    # UTM
    for m in RE_UTM.finditer(texto):
        zona, banda, este, norte = m.groups()
        resultados.append({"formato": "UTM", "texto": m.group(0).strip(),
                           "zona": f"{zona}{banda}", "este": float(este),
                           "norte": float(norte), "pos": m.start()})

    return sorted(resultados, key=lambda r: r["pos"])


def buscar_coordenadas(texto, buscar="", ventana=200):
    """Si `buscar` está vacío, busca en todo el texto.
    Si no, busca coordenadas justo después de cada aparición exacta de `buscar`."""
    if not buscar:
        return extraer_coordenadas(texto)

    resultados = []
    inicio = 0
    while (idx := texto.find(buscar, inicio)) != -1:
        desde = idx + len(buscar)
        siguiente = texto.find(buscar, desde)
        hasta = min(desde + ventana, siguiente if siguiente != -1 else len(texto))
        for r in extraer_coordenadas(texto[desde:hasta])[:1]:   # la primera tras la frase
            r["pos"] += desde
            resultados.append(r)
        inicio = desde
    return resultados

In [ ]:
# === 3. Leer el PDF ===
def leer_pdf(ruta):
    from pypdf import PdfReader
    lector = PdfReader(ruta)
    paginas = []
    for n, pagina in enumerate(lector.pages, 1):
        paginas.append((n, pagina.extract_text() or ""))
    return paginas


if RUTA_PDF:
    PAGINAS = leer_pdf(RUTA_PDF)
    total = sum(len(t) for _, t in PAGINAS)
    print(f"PDF leído: {len(PAGINAS)} página(s), {total} caracteres de texto.")
    if total == 0:
        print("AVISO: el PDF no tiene texto seleccionable (probablemente es escaneado). Haría falta OCR.")
else:
    PAGINAS = [(1, TEXTO)]
    print("Usando el TEXTO escrito en la celda 1.")

In [ ]:
# === 4. Ejecutar y mostrar resultados ===
coordenadas = []
for num_pagina, texto_pagina in PAGINAS:
    for c in buscar_coordenadas(texto_pagina, BUSCAR, VENTANA):
        c["pagina"] = num_pagina
        coordenadas.append(c)

texto_completo = "\n".join(t for _, t in PAGINAS)
if BUSCAR and BUSCAR not in texto_completo:
    print(f'No se encontró el texto "{BUSCAR}".')
elif not coordenadas:
    print("No se encontraron coordenadas.")
else:
    print(f"Se encontraron {len(coordenadas)} coordenada(s):\n")
    for n, c in enumerate(coordenadas, 1):
        pag = f"pág. {c['pagina']}"
        if c["formato"] == "UTM":
            print(f"{n}. {pag} [UTM]     zona {c['zona']}  este={c['este']}  norte={c['norte']}   <- {c['texto']}")
        else:
            print(f"{n}. {pag} [{c['formato']:7}] lat={c['lat']:.6f}  lon={c['lon']:.6f}   <- {c['texto']}")